# CSE 144 Final Project — Transfer Learning Challenge
**Models:** EfficientNet-B0, B3, B4 · ResNet50 · ViT-B/16
**Best Kaggle Score:** 0.772

## Setup
1. Mount Google Drive
2. Set `TRAIN_DIR` and `TEST_DIR` to your data paths
3. Run all cells in order

**Runtime:** Runtime → Change runtime type → T4 GPU

## 1. Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
TRAIN_DIR = '/content/drive/MyDrive/CSE144_Project/train'
TEST_DIR  = '/content/drive/MyDrive/CSE144_Project/test'

print('Train classes:', sorted(os.listdir(TRAIN_DIR))[:5])
print('Test images:  ', sorted(os.listdir(TEST_DIR))[:5])
print('Num test files:', len(os.listdir(TEST_DIR)))

## 2. Imports & Reproducibility

In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets.folder import default_loader
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 3. Data Transforms

In [ ]:
# Training: strong augmentation to combat overfitting on small dataset
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Validation/Test: no augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Transforms defined.')

## 4. Dataset & DataLoaders

> **Label mapping fix:** `ImageFolder` sorts folders alphabetically so `'10'` would get label `2` instead of `10`. We override `class_to_idx` to use numeric sorting so folder `'N'` always maps to label `N`.

In [ ]:
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)

# Fix label mapping: folder '10' -> label 10 (not label 2)
full_dataset.class_to_idx = {cls: int(cls) for cls in full_dataset.classes}
full_dataset.targets = [full_dataset.class_to_idx[s[0].split('/')[-2]] for s in full_dataset.samples]
full_dataset.samples = [(s[0], full_dataset.class_to_idx[s[0].split('/')[-2]]) for s in full_dataset.samples]

# 80/20 split used during model selection experiments
n_val   = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val
train_set, val_set = random_split(full_dataset, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(SEED))
val_set.dataset.transform = val_transform

val_loader  = DataLoader(val_set,      batch_size=32, shuffle=False, num_workers=2)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True,  num_workers=2)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Classes: {len(full_dataset.classes)}')
print(f'Train: {n_train} | Val: {n_val}')
print(f'Label check: {list(full_dataset.class_to_idx.items())[:5]}')

## 5. Training Helper

In [ ]:
def train_model(model, loader, epochs, save_path):
    """Train on full dataset and save final weights."""
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
        scheduler.step()
        print(f'Epoch {epoch+1:02d} | Loss: {total_loss/len(loader):.3f} | Train: {correct/len(loader.dataset):.3f}')

    torch.save(model.state_dict(), save_path)
    print(f'Saved -> {save_path}')


def load_efficientnet(variant, path):
    m = getattr(models, f'efficientnet_{variant}')(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 100)
    m.load_state_dict(torch.load(path))
    return m.to(device).eval()

def load_resnet(variant, path):
    m = getattr(models, f'resnet{variant}')(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 100)
    m.load_state_dict(torch.load(path))
    return m.to(device).eval()

def load_vit(path):
    m = models.vit_b_16(weights=None)
    m.heads.head = nn.Linear(m.heads.head.in_features, 100)
    m.load_state_dict(torch.load(path))
    return m.to(device).eval()

BASE = '/content/drive/MyDrive/CSE144_Project'
print('Helpers defined.')

## 6. Train All 5 Models (Full Dataset)

All models trained on the full 1079-image dataset for 50 epochs.
- Optimizer: AdamW (lr=1e-4, weight_decay=1e-2)
- Scheduler: CosineAnnealingLR
- Loss: CrossEntropyLoss with label_smoothing=0.1

In [ ]:
print('=== EfficientNet-B0 ===')
m = models.efficientnet_b0(weights='IMAGENET1K_V1')
m.classifier[1] = nn.Linear(m.classifier[1].in_features, 100)
train_model(m.to(device), full_loader, epochs=50, save_path=f'{BASE}/best_model_b0_full.pth')

In [ ]:
print('=== EfficientNet-B3 ===')
m = models.efficientnet_b3(weights='IMAGENET1K_V1')
m.classifier[1] = nn.Linear(m.classifier[1].in_features, 100)
train_model(m.to(device), full_loader, epochs=50, save_path=f'{BASE}/best_model_b3_full.pth')

In [ ]:
print('=== EfficientNet-B4 ===')
m = models.efficientnet_b4(weights='IMAGENET1K_V1')
m.classifier[1] = nn.Linear(m.classifier[1].in_features, 100)
train_model(m.to(device), full_loader, epochs=50, save_path=f'{BASE}/best_model_b4_full.pth')

In [ ]:
print('=== ResNet50 ===')
m = models.resnet50(weights='IMAGENET1K_V2')
m.fc = nn.Linear(m.fc.in_features, 100)
train_model(m.to(device), full_loader, epochs=50, save_path=f'{BASE}/best_model_r50_full.pth')

In [ ]:
print('=== ViT-B/16 ===')
m = models.vit_b_16(weights='IMAGENET1K_V1')
m.heads.head = nn.Linear(m.heads.head.in_features, 100)
train_model(m.to(device), full_loader, epochs=50, save_path=f'{BASE}/best_model_vit_full.pth')

## 7. Load Trained Models

In [ ]:
model_b0  = load_efficientnet('b0', f'{BASE}/best_model_b0_full.pth')
model_b3  = load_efficientnet('b3', f'{BASE}/best_model_b3_full.pth')
model_b4  = load_efficientnet('b4', f'{BASE}/best_model_b4_full.pth')
model_r50 = load_resnet('50',       f'{BASE}/best_model_r50_full.pth')
model_vit = load_vit(              f'{BASE}/best_model_vit_full.pth')

all_models = [model_b0, model_b3, model_b4, model_r50, model_vit]
print(f'Loaded {len(all_models)} models.')

## 8. Generate Submission

5-model ensemble: average softmax probabilities across all models.
**Best Kaggle score: 0.772**

In [ ]:
test_ids = [f'{i}.jpg' for i in range(1036)]  # Kaggle expects 1036 predictions
predictions = []

with torch.no_grad():
    for fname in test_ids:
        img = default_loader(os.path.join(TEST_DIR, fname))
        img_tensor = val_transform(img).unsqueeze(0).to(device)
        probs = sum(torch.softmax(m(img_tensor), dim=1) for m in all_models) / len(all_models)
        predictions.append({'ID': fname, 'Label': probs.argmax(1).item()})

df = pd.DataFrame(predictions)
df.to_csv(f'{BASE}/submission_5model.csv', index=False)
print(f'Total predictions: {len(df)}')
print(df.head(10))